# 📓 Lab 04 — Secuencias: tokenización, embeddings y RNN/LSTM

**Curso:** Deep Learning: de los fundamentos a los Transformers · **Sesión 3**
**Material del aula:** [Sesión 3 — Secuencias y Transformers](../sesiones/03-secuencias-transformers.md)

En este notebook: texto → tokens → IDs → embeddings → RNN/LSTM, y la
demostración empírica del vanishing gradient que motiva la attention.

In [ ]:
from __future__ import annotations

import sys
sys.path.append('..')

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

from src.utils import seed_everything, detectar_dispositivo
from src.data import tokenizar_basico, construir_vocabulario, collate_texto

seed_everything(42)
device = detectar_dispositivo()
print('Dispositivo:', device)

## 1. De texto a IDs: el pipeline mínimo

Todo modelo de lenguaje empieza igual: texto → tokens → IDs enteros.
Usamos el tokenizador simple del curso para ENTENDER el mecanismo antes de
usar los subword tokenizers de Hugging Face (Lab 06).

In [ ]:
corpus = [
    'el gato come pescado fresco',
    'el perro come carne',
    'la película no es buena',      # ¡la negación importa!
    'la película es buena',
    'me encanta este curso de deep learning',
]

# Tokenizar y construir vocabulario (en un proyecto real: SOLO con train)
itos, stoi = construir_vocabulario(corpus, min_frecuencia=1)
print(f'Vocabulario: {len(itos)} tokens')
print('Primeros 12:', itos[:12])   # <pad>=0 (relleno, ver sección 3) y
                                   # <unk>=1 (token desconocido) siempre primero

frase = 'el gato no come carne'
tokens = tokenizar_basico(frase)
ids = [stoi.get(t, 1) for t in tokens]   # 1 = <unk> para desconocidos
print(f'\n{frase!r}')
print('tokens:', tokens)
print('ids   :', ids)

## 2. Embeddings: cada ID se vuelve un vector entrenable

`nn.Embedding` es una matriz $E \in \mathbb{R}^{|V| \times d}$ donde la fila
$i$ es el vector del token $i$. Es un lookup **diferenciable**: los vectores
se ajustan por backpropagation igual que cualquier peso.

In [ ]:
d_model = 16
embedding = nn.Embedding(num_embeddings=len(itos), embedding_dim=d_model,
                         padding_idx=0)   # la fila de <pad> se congela en ceros

ids_tensor = torch.tensor(ids)
vectores = embedding(ids_tensor)
print('ids     :', ids_tensor.shape)      # (T,)
print('vectores:', vectores.shape)        # (T, d_model)

# Similitud coseno: mide el ángulo entre dos vectores — 1 ≈ misma dirección
# (significados parecidos), 0 ≈ sin relación, −1 ≈ opuestos.
# Entre dos tokens (antes de entrenar: ruido puro;
# después de entrenar en una tarea real: semántica distribuida)
def coseno(a, b):
    return torch.nn.functional.cosine_similarity(a, b, dim=0).item()

v_gato = embedding(torch.tensor(stoi['gato']))
v_perro = embedding(torch.tensor(stoi['perro']))
print(f"\ncos(gato, perro) sin entrenar: {coseno(v_gato, v_perro):.3f}")

## 3. Padding y máscaras en un batch real

Secuencias de longitudes distintas → rellenar al máximo del batch
(**dynamic padding**) + máscara booleana. Ver `collate_texto` en
[`src/data.py`](../src/data.py).

In [ ]:
batch_crudo = [
    (torch.tensor([stoi.get(t, 1) for t in tokenizar_basico(s)]), i % 2)
    for i, s in enumerate(corpus[:3])
]

padded, mask, labels = collate_texto(batch_crudo)
print('padded (B, T_max):'); print(padded)
print('\nmask (True = token real):'); print(mask)

## 4. RNN paso a paso

$$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)$$

Implementamos la recurrencia A MANO para ver el estado actualizarse.

In [ ]:
class RNNManual(nn.Module):
    """Celda RNN mínima, para ver la recurrencia sin envoltorios."""

    def __init__(self, d_in: int, d_hidden: int) -> None:
        super().__init__()
        self.W_xh = nn.Linear(d_in, d_hidden)      # entrada → estado
        self.W_hh = nn.Linear(d_hidden, d_hidden)  # estado → estado

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # x: (B, T, d_in) → estados: (B, T, d_hidden)
        B, T, _ = x.shape
        h = torch.zeros(B, self.W_hh.in_features, device=x.device)
        estados = []
        for t in range(T):                          # ← el bucle secuencial:
            h = torch.tanh(self.W_xh(x[:, t]) + self.W_hh(h))  # la debilidad
            estados.append(h)                       #    estructural de la RNN
        return torch.stack(estados, dim=1), h

rnn = RNNManual(d_in=d_model, d_hidden=32)
x_seq = embedding(padded)                # (B, T, d_model)
estados, h_final = rnn(x_seq)
print('estados:', estados.shape)         # (B, T, 32)
print('h_final:', h_final.shape)         # (B, 32) — resumen de la secuencia

## 5. El experimento clave: vanishing gradients

Medimos cuánto gradiente llega desde la loss (en el último paso) hasta CADA
paso temporal. Si la RNN no puede "recordar" pasos lejanos, el gradiente
hacia ellos debe ser diminuto.

In [ ]:
seed_everything(0)
T = 30
rnn_test = RNNManual(d_in=8, d_hidden=32)

x = torch.randn(1, T, 8, requires_grad=True)
estados, h_final = rnn_test(x)
loss = h_final.sum()
loss.backward()

# Norma del gradiente que llegó a cada paso temporal de la ENTRADA
normas = x.grad.squeeze(0).norm(dim=1).detach().numpy()

plt.figure(figsize=(8, 4))
plt.semilogy(range(1, T + 1), normas[::-1], 'o-')
plt.xlabel('distancia temporal desde la loss')
plt.ylabel('‖gradiente‖ (escala log)')
plt.title('Vanishing gradient: la señal se apaga con la distancia')
plt.show()

# Lectura: cada paso multiplica el gradiente por una matriz de derivadas
# (un 'factor de amplificación', formalmente el Jacobiano);
# el producto de muchos factores <1 tiende a cero exponencialmente.

## 6. LSTM: la solución con compuertas

`nn.LSTM` implementa las 6 ecuaciones de la
[Sesión 3 §2](../sesiones/03-secuencias-transformers.md#2-rnn-la-primera-respuesta-al-problema-secuencial).
La celda de memoria $c_t$ tiene un camino ADITIVO
($c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$) que preserva el gradiente.

In [ ]:
lstm = nn.LSTM(input_size=d_model, hidden_size=32, batch_first=True)

salida, (h_n, c_n) = lstm(x_seq)
print('salida (todos los estados):', salida.shape)   # (B, T, 32)
print('h_n (último estado):       ', h_n.shape)      # (1, B, 32)
print('c_n (celda de memoria):    ', c_n.shape)      # (1, B, 32)

# Clasificador many-to-one (muchas entradas → una sola predicción) típico:
# Embedding → LSTM → último estado válido (¡cuidado con el padding!) → Linear
class ClasificadorLSTM(nn.Module):
    """Embedding → LSTM → último estado REAL (según máscara) → head."""

    def __init__(self, vocab_size: int, d_model: int = 32,
                 d_hidden: int = 64, num_classes: int = 2) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.lstm = nn.LSTM(d_model, d_hidden, batch_first=True)
        self.head = nn.Linear(d_hidden, num_classes)

    def forward(self, ids: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        salida, _ = self.lstm(self.embedding(ids))       # (B, T, d_hidden)
        # Índice del último token REAL de cada secuencia (no del padding)
        ultimos = mask.sum(dim=1) - 1                     # (B,)
        estado_final = salida[torch.arange(len(ids)), ultimos]  # (B, d_hidden)
        return self.head(estado_final)

clf = ClasificadorLSTM(vocab_size=len(itos))
logits = clf(padded, mask)
print('\nlogits:', logits.shape)   # (B, 2)

## 7. 🎯 Reto y transición

1. Cambia `T = 30` por `T = 60` en el experimento de gradientes. ¿Empeora?
2. Repite el experimento con `nn.LSTM`: registra la norma del gradiente
   hacia `x` y compara las curvas. ¿La LSTM preserva mejor la señal?
3. Aun así, la LSTM procesa token a token (bucle secuencial) y comprime
   todo en un estado. En el [Lab 05](05_attention_from_scratch.ipynb)
   la attention elimina ambas limitaciones.

**Commit sugerido:** `feat: complete sequences lab`